# Notebook 06: Real Speculative Decoding + End-to-End Latency Budget & Cost Capstone

`[REAL]` + `[COMPUTED FROM REAL DATA]` + `[SIMULATION]` Companion to Modules 07, 08, and 09. Real speculative decoding on the RTX 4060 using a real INT4-quantized copy of `Qwen2.5-0.5B-Instruct` as the draft model against the real FP16 model as the target/verifier (a real, legitimate self-speculative-decoding pattern: same weights, lower precision, genuinely cheaper per forward pass per Notebook 04's memory findings).

**Per the signed-off plan:** real acceptance rate and real end-to-end speedup are measured and reported as two genuinely separate numbers (Sections 1-2), never conflated -- mirroring Module 07's own two-step formula discipline. If speculative decoding had proven infeasible on this real hardware, that would be reported honestly instead of substituting an unrelated comparison; it proved feasible, so no such fallback is needed here.

In [1]:
import time
import math
import logging
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
EOS_IDS = set(tokenizer.eos_token_id) if isinstance(tokenizer.eos_token_id, list) else {tokenizer.eos_token_id}

target_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(DEVICE)
target_model.eval()

bnb_int4_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
draft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_int4_config, device_map=DEVICE)
draft_model.eval()
print("Target (FP16) and draft (INT4) models loaded -- real speculative decoding is feasible on this hardware.")

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  46%|████▌     | 133/290 [00:00<00:00, 1329.48it/s]

Loading weights:  92%|█████████▏| 266/290 [00:00<00:00, 1283.44it/s]

...

Loading weights:  72%|███████▏  | 208/290 [00:00<00:00, 282.16it/s]

Loading weights:  84%|████████▍ | 243/290 [00:01<00:00, 296.28it/s]

Loading weights:  94%|█████████▍| 274/290 [00:01<00:00, 295.46it/s]

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 233.92it/s]

Target (FP16) and draft (INT4) models loaded -- real speculative decoding is feasible on this hardware.


## 1. Real Acceptance Rate (Manual, Verified-Correct Speculative Decoding Loop)

`[REAL]` Transformers' built-in `assistant_model=` API doesn't directly expose a per-round acceptance count, so this notebook implements a manual, transparent speculative-decoding loop instead -- draft proposes `k` tokens, target verifies all `k` in one real forward pass by comparing `argmax` at each position, accepting the longest real matching prefix. **Correctness was independently verified** before trusting these results: run against a real prompt, this loop's output was confirmed byte-for-byte identical to standard `model.generate()` greedy output for the same length (a real bug was caught and fixed in the process -- see the note below).

In [2]:
# Real bug found and fixed during development: the model's default generation_config sets
# repetition_penalty=1.1, which generate() applies even under do_sample=False -- so "greedy" via
# generate() is NOT plain argmax(logits) unless repetition_penalty is explicitly reset to 1.0. This
# loop and every generate() call below pass repetition_penalty=1.0 explicitly for a fair, exact-argmax
# comparison; the manual loop's output was verified to exactly match generate()'s output only after
# this fix (before the fix, they diverged -- a real, caught discrepancy, not assumed away).

def manual_speculative_decode(prompt_ids, k, n_target_tokens):
    context = prompt_ids.clone()
    total_accepted, total_proposed, n_rounds = 0, 0, 0
    start_len = context.shape[1]
    while context.shape[1] - start_len < n_target_tokens:
        with torch.no_grad():
            draft_out = draft_model.generate(context, max_new_tokens=k, do_sample=False,
                                              repetition_penalty=1.0, pad_token_id=tokenizer.eos_token_id)
        draft_tokens = draft_out[0, context.shape[1]:]
        k_actual = draft_tokens.shape[0]
        if k_actual == 0:
            break
        combined = torch.cat([context, draft_tokens.unsqueeze(0)], dim=1)
        with torch.no_grad():
            target_out = target_model(combined, use_cache=False)
        logits = target_out.logits[0]
        ctx_len = context.shape[1]
        accepted = 0
        for i in range(k_actual):
            pred = logits[ctx_len - 1 + i].argmax().item()
            if pred == draft_tokens[i].item():
                accepted += 1
            else:
                break
        bonus_pred = logits[ctx_len - 1 + accepted].argmax().item()
        new_tokens = draft_tokens[:accepted].tolist() + [bonus_pred]
        context = torch.cat([context, torch.tensor([new_tokens], device=context.device)], dim=1)
        total_accepted += accepted
        total_proposed += k_actual
        n_rounds += 1
        if bonus_pred in EOS_IDS:
            break
    return context, total_accepted, total_proposed, n_rounds

PROMPT = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Explain the concept of gravity in simple terms."}],
    tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(DEVICE)
K_DRAFT = 4
N_TARGET_TOKENS = 64

spec_out_ids, accepted, proposed, rounds = manual_speculative_decode(input_ids, K_DRAFT, N_TARGET_TOKENS)
real_alpha = accepted / proposed
print(f"Real accepted: {accepted}, real proposed: {proposed}, rounds: {rounds}")
print(f"Real empirical acceptance rate (alpha): {real_alpha:.4f}")

n_new = spec_out_ids.shape[1] - input_ids.shape[1]
with torch.no_grad():
    greedy_check = target_model.generate(input_ids, max_new_tokens=n_new, do_sample=False,
                                          repetition_penalty=1.0, pad_token_id=tokenizer.eos_token_id)
match_len = min(greedy_check.shape[1], spec_out_ids.shape[1])
outputs_match = torch.equal(greedy_check[0, :match_len], spec_out_ids[0, :match_len])
print(f"Correctness check -- spec-decode output matches standard greedy output: {outputs_match}")

print("\n(pending real interpretation)")

Real accepted: 40, real proposed: 100, rounds: 25
Real empirical acceptance rate (alpha): 0.4000


Correctness check -- spec-decode output matches standard greedy output: True

(pending real interpretation)


**Real result:** `40` accepted out of `100` proposed draft tokens across `25` rounds — a real empirical acceptance rate of **`α = 0.4000`**. The correctness check confirms `True`: this manual loop's output is byte-for-byte identical to standard `generate()` greedy output, so `α=0.4000` reflects a real, verified-correct measurement, not an artifact of a buggy comparison.

## 2. Real End-to-End Speedup (Kept Separate From Acceptance Rate, Per Module 07's Discipline)

`[REAL]` Measuring real wall-clock time for this manual speculative-decoding loop versus real wall-clock time for standard greedy decoding of the same real output length -- a **separate, distinct real measurement** from Section 1's acceptance rate, not derived from it.

In [3]:
def timed_run(fn, n_repeats=3, n_warmup=1):
    for _ in range(n_warmup):
        fn()
        torch.cuda.synchronize()
    samples = []
    for _ in range(n_repeats):
        torch.cuda.synchronize()
        start = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        samples.append(time.perf_counter() - start)
    return sorted(samples)[len(samples) // 2]

def run_spec_decode():
    manual_speculative_decode(input_ids, K_DRAFT, N_TARGET_TOKENS)

def run_standard_greedy():
    with torch.no_grad():
        target_model.generate(input_ids, max_new_tokens=N_TARGET_TOKENS, min_new_tokens=N_TARGET_TOKENS,
                               do_sample=False, repetition_penalty=1.0, pad_token_id=tokenizer.eos_token_id)

spec_decode_median_s = timed_run(run_spec_decode)
standard_greedy_median_s = timed_run(run_standard_greedy)
real_speedup = standard_greedy_median_s / spec_decode_median_s

print(f"Manual spec-decode loop: {spec_decode_median_s*1000:.1f}ms median")
print(f"Standard greedy decode:  {standard_greedy_median_s*1000:.1f}ms median")
print(f"Real measured speedup: {real_speedup:.3f}x")

formula_expected_accepted = (1 - real_alpha ** (K_DRAFT + 1)) / (1 - real_alpha) if real_alpha < 1 else K_DRAFT + 1
print(f"\nModule 07's formula, evaluated at this real measured alpha={real_alpha:.4f}, k={K_DRAFT}: "
      f"E[accepted]={formula_expected_accepted:.3f} tokens/round")

print("\n(pending real interpretation)")

Manual spec-decode loop: 5063.1ms median
Standard greedy decode:  1850.0ms median
Real measured speedup: 0.365x

Module 07's formula, evaluated at this real measured alpha=0.4000, k=4: E[accepted]=1.650 tokens/round

(pending real interpretation)


**Real result — a genuine slowdown, reported as-is:** the manual spec-decode loop took `5063.1ms` median versus standard greedy's `1850.0ms` — real measured speedup is **`0.365x`**, i.e., this implementation was real, measured `≈2.74x` *slower*, not faster. This does **not** contradict Section 1's real `α=0.4000` result or Module 07's formula (which independently predicts `E[accepted]=1.650` tokens/round — closely matching this run's actual real average of `40/25=1.6` accepted tokens/round). What it demonstrates instead is exactly the real, separate cost this notebook's implementation carries and Module 07 itself warns about: this manual loop calls `draft_model.generate()` from scratch every round (no draft KV-cache reuse across rounds) and runs the target's verification with `use_cache=False` (recomputing the *entire* growing context from scratch every round, not just the new tokens) — real, substantial, avoidable overhead that a production speculative-decoding implementation (which reuses KV caches on both sides) would not pay. **Acceptance rate and speedup are consequently very different real findings**, exactly per the plan's discipline: a real `α=0.4000` is a genuinely decent acceptance rate, but this specific, deliberately-transparent-over-optimized implementation's real wall-clock cost doesn't reflect that — a caveat worth remembering whenever comparing a "correctness-first" reference implementation's timing against a production-grade one.

## 3. Capstone: Real-Data Latency Budget & Cost (Module 09)

`[COMPUTED FROM REAL DATA]` Filling in Module 09's latency-budget worked example with genuinely real measured prefill/decode numbers from Notebook 01, replacing that module's illustrative 300ms/1600ms figures. Queue-wait and network/serialization remain the module's original **stated assumptions** (150ms, 50ms) -- there is no real network stack or request queue to measure in a single-notebook, single-machine context, and this section does not claim otherwise.

In [4]:
# Real measured values, quoted directly from Notebook 01's own executed output:
REAL_TTFT_512_MS = 156.49   # real median TTFT at 512 real prompt tokens
REAL_TPOT_FLOOR_MS = 138.058  # real median TPOT floor (128-token real generation run)
ASSUMED_QUEUE_WAIT_MS = 150  # stated assumption, not measurable here
ASSUMED_NETWORK_MS = 50      # stated assumption, not measurable here
N_OUTPUT_TOKENS = 100

real_decode_ms = REAL_TPOT_FLOOR_MS * N_OUTPUT_TOKENS
total_budget_ms = ASSUMED_QUEUE_WAIT_MS + REAL_TTFT_512_MS + real_decode_ms + ASSUMED_NETWORK_MS
decode_share_pct = real_decode_ms / total_budget_ms * 100

print(f"Queue (assumed): {ASSUMED_QUEUE_WAIT_MS}ms, Prefill (REAL): {REAL_TTFT_512_MS}ms, "
      f"Decode (REAL, {N_OUTPUT_TOKENS} tok): {real_decode_ms:.2f}ms, Network (assumed): {ASSUMED_NETWORK_MS}ms")
print(f"Total budget: {total_budget_ms:.2f}ms, Decode share: {decode_share_pct:.2f}%")

GPU_COST_RATE_PER_HOUR = 2.00  # illustrative representative rate, not this laptop's real billed cost
gpu_time_s = (REAL_TTFT_512_MS + real_decode_ms) / 1000
cost_request = gpu_time_s * GPU_COST_RATE_PER_HOUR / 3600
cost_per_token = cost_request / N_OUTPUT_TOKENS
print(f"\nReal GPU time: {gpu_time_s:.5f}s, Cost/request: ${cost_request:.6f}, Cost/token: ${cost_per_token:.8f}")

print("\n(pending real interpretation)")

Queue (assumed): 150ms, Prefill (REAL): 156.49ms, Decode (REAL, 100 tok): 13805.80ms, Network (assumed): 50ms
Total budget: 14162.29ms, Decode share: 97.48%

Real GPU time: 13.96229s, Cost/request: $0.007757, Cost/token: $0.00007757

(pending real interpretation)


**Real result:** filling Module 09's budget with genuinely measured prefill (`156.49ms`) and decode (`13,805.80ms` for 100 real output tokens, from the real TPOT floor) alongside the module's stated queue/network assumptions gives a total budget of `14,162.29ms`, with decode now accounting for **`97.48%`** of the total — even more extreme than Module 09's original illustrative `76.2%` figure, because this real small-model decode cost, multiplied across 100 real tokens, dwarfs the small fixed assumed queue/network terms. The real GPU-time-based cost model gives `$0.007757` per request and `$0.00007757` per token at the illustrative `$2.00/hr` rate — real numbers derived from genuine measured GPU-active time, not an API price sheet, exactly per Module 09's cost-model discipline.

## 4. Simulation: Least-Loaded Routing Using Real-Grounded Service Times (Module 08)

`[SIMULATION]` Converting Notebook 05's real measured generation lengths (`[8, 10, 2, 7, 11, 80, 80, 2]`) into real-grounded service-time estimates (length x Notebook 01's real TPOT floor), then simulating least-loaded routing across 3 replicas. **This is a simulation, not a real multi-GPU deployment** -- no actual second or third GPU exists in this environment; only the input service-time estimates are genuinely real-data-grounded.

In [5]:
REAL_GROUNDED_LENGTHS = [8, 10, 2, 7, 11, 80, 80, 2]  # from Notebook 05's real bs=8 run
service_times_ms = [L * REAL_TPOT_FLOOR_MS for L in REAL_GROUNDED_LENGTHS]
print(f"Real-grounded per-request service times (ms): {[round(s, 1) for s in service_times_ms]}")

def least_loaded_route(replica_loads, service_time):
    target = min(replica_loads, key=lambda r: replica_loads[r])
    replica_loads[target] += service_time
    return target

replicas = {"replica_0": 0.0, "replica_1": 0.0, "replica_2": 0.0}
assignments = {}
for i, st in enumerate(service_times_ms):
    target = least_loaded_route(replicas, st)
    assignments[f"req_{i}"] = target
    rounded_loads = {k: round(v, 1) for k, v in replicas.items()}
    print(f"req_{i} (service_time={st:.1f}ms) -> {target}, loads now: {rounded_loads}")

final_spread = max(replicas.values()) - min(replicas.values())
print(f"\nFinal simulated load spread across 3 replicas: {final_spread:.1f}ms")
print("\n(pending real interpretation)")

Real-grounded per-request service times (ms): [1104.5, 1380.6, 276.1, 966.4, 1518.6, 11044.6, 11044.6, 276.1]
req_0 (service_time=1104.5ms) -> replica_0, loads now: {'replica_0': 1104.5, 'replica_1': 0.0, 'replica_2': 0.0}
req_1 (service_time=1380.6ms) -> replica_1, loads now: {'replica_0': 1104.5, 'replica_1': 1380.6, 'replica_2': 0.0}
req_2 (service_time=276.1ms) -> replica_2, loads now: {'replica_0': 1104.5, 'replica_1': 1380.6, 'replica_2': 276.1}
req_3 (service_time=966.4ms) -> replica_2, loads now: {'replica_0': 1104.5, 'replica_1': 1380.6, 'replica_2': 1242.5}
req_4 (service_time=1518.6ms) -> replica_0, loads now: {'replica_0': 2623.1, 'replica_1': 1380.6, 'replica_2': 1242.5}
req_5 (service_time=11044.6ms) -> replica_2, loads now: {'replica_0': 2623.1, 'replica_1': 1380.6, 'replica_2': 12287.2}
req_6 (service_time=11044.6ms) -> replica_1, loads now: {'replica_0': 2623.1, 'replica_1': 12425.2, 'replica_2': 12287.2}
req_7 (service_time=276.1ms) -> replica_0, loads now: {'replica_

## 5. Real Interpretation

`[SIMULATION]` Two of the eight real-grounded service times (`11,044.6ms` each, from Notebook 05's two straggler sequences that never hit EOS) dominate this simulation entirely — the least-loaded router correctly places one on `replica_1` and the other on `replica_2` rather than stacking both on the same replica, but with only two "giant" requests and three replicas, one replica (`replica_0`) inevitably ends up carrying only small requests while the other two each absorb one giant one. The resulting real, simulated load spread — `9,526.0ms` between the busiest and least-busy replica — is large not because the router is inefficient, but because the underlying real-grounded workload itself is extremely skewed (a real `40x` spread between the shortest and longest generation, carried over directly from Notebook 05's genuine straggler finding). **This is a `[SIMULATION]`**: no second or third real GPU exists in this environment; the routing logic and the resulting load spread are simulated, even though the service-time inputs are real-data-grounded. The finding it illustrates is genuine, though: real request-length variance (Module 06) directly determines how well even a correct least-loaded router (Module 08) can balance real replica load — a router cannot fix workload skew it wasn't given the data to smooth out.